In [7]:
# TESTES 1 e 2 - TRADE-OFFS no README.md

In [8]:
import pandas as pd
from api_requests import baixar_e_descompactar

In [9]:
from api_requests import baixar_e_descompactar

api_url = [
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/1T2025.zip',
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/2T2025.zip',
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/3T2025.zip',
    #'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_acreditadas/operadoras_acreditadas.csv',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_ativas/Relatorio_cadop.csv',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_canceladas/Relatorio_cadop_canceladas.csv',
    #'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_e_prestadores_nao_hospitalares/operadoras_e_prestadores_nao_hospitalares.zip'
]

baixar_e_descompactar(api_url)


Baixando 1T2025.zip...


Descompactando 1T2025.zip
Baixando 2T2025.zip...
Descompactando 2T2025.zip
Baixando 3T2025.zip...
Descompactando 3T2025.zip
Baixando Relatorio_cadop.csv...
Baixando Relatorio_cadop_canceladas.csv...


In [10]:
import pandas as pd
import os

#apenas arquivos dos trimestres
paths = [
    os.path.join("data", f)
    for f in os.listdir("data")
    if f.endswith(".csv") and f.startswith(("1T", "2T", "3T"))
]

def le_exibe_colunas(path):
    chunk = next(pd.read_csv(path, sep=';', encoding='latin1', chunksize=100_000))
    print(chunk.columns.tolist())
    
    
for path in paths:
    print(f"\nArquivo: {path}")
    le_exibe_colunas(path)

def trimestre_filtrados(path, valor, coluna='DESCRICAO'):
    for chunk in pd.read_csv(path, sep=';', encoding='latin1', chunksize=100_000):
        mask = (
            chunk[coluna]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.contains(valor.upper(), na=False, regex=True)
        )
        if mask.any():
            yield chunk.loc[mask]
            
df_eventos = pd.concat(
    (
        chunk
        for path in paths
        for chunk in trimestre_filtrados(path, r'(?=.*EVENT)(?=.*SINISTR)')
    ),
    ignore_index=True
)

df_eventos.head()




Arquivo: data/1T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']

Arquivo: data/2T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']

Arquivo: data/3T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']


,DATA,REG_ANS,CD_CONTA_CONTABIL,DESCRICAO,VL_SALDO_INICIAL,VL_SALDO_FINAL
0,2025-01-01,316849,131719011,DepÃ³sitos Judiciais - Eventos / Sinistros,"289349,17","292907,23"
1,2025-01-01,316849,21111203,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,0,0
2,2025-01-01,316849,23111202,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,"203169,04","199868,05"
3,2025-01-01,316849,231112022,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,"203169,04","199868,05"
4,2025-01-01,316903,231111021,ProvisÃ£o de Eventos/Sinistros a Liquidar para...,25535,25535


In [11]:
# Baixei todos os arquivos de operadoras inicialmente. No desafio 1.3 é necessario criar um .csv com colunas: 
# CNPJ , RazaoSocial , Trimestre , Ano , ValorDespesaTotal. Através da análise dos arquivos, notei que os arquivos de operadoras de plano de saúde são
# os que devem ser utilizados na análise, pois operadoras não hospitalares e operadoras acreditadas não possuem informações necessárias. Colunas das mesmas:
# ['Reg ANS', 'Operadora', 'Nivel_Acreditacao', 'Inicio_Validade', 'Fim_Validade', 'Prazo_validade', 'Resolucao_Normativa', 'Entidade_Acreditadora', 'Reacreditada', 'Data_da_Primeira_Acreditacao', 'Total de Anos de Acredita\x87Æo*']
# ['REGISTRO_OPERADORA', 'NM_OPERADORA', 'GR_MODALIDADE', 'ID_ESTABELECIMENTO_SAUDE', 'CD_CNPJ_ESTB_SAUDE', 'CD_CNES', 'NM_ESTABELECIMENTO_SAUDE', 'DE_CLAS_ESTB_SAUDE', 'DE_TIPO_PRESTADOR', 'LG_URGENCIA_EMERGENCIA', 'DE_TIPO_CONTRATO', 'DE_DISPONIBILIDADE', 'CD_MUNICIPIO', 'NM_MUNICIPIO', 'SG_UF', 'NM_REGIAO', 'DT_VINCULO_OPERADORA_INICIO', 'DT_VINCULO_OPERADORA_FIM', 'COMPETENCIA', 'DT_ATUALIZACAO']

In [12]:
operadoras = [
    os.path.join('data', f)
    for f in os.listdir('data')
    if f.startswith(("Relatorio")) and f.endswith(".csv")    
]

for operador in operadoras:
    print(f"Arquivo: {operador} ")
    le_exibe_colunas(operador)


Arquivo: data/Relatorio_cadop.csv 
['REGISTRO_OPERADORA', 'CNPJ', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'Logradouro', 'Numero', 'Complemento', 'Bairro', 'Cidade', 'UF', 'CEP', 'DDD', 'Telefone', 'Fax', 'Endereco_eletronico', 'Representante', 'Cargo_Representante', 'Regiao_de_Comercializacao', 'Data_Registro_ANS']
Arquivo: data/Relatorio_cadop_canceladas.csv 


['REGISTRO_OPERADORA', 'CNPJ', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'Logradouro', 'Numero', 'Complemento', 'Bairro', 'Cidade', 'UF', 'CEP', 'DDD', 'Telefone', 'Fax', 'Endereco_eletronico', 'Representante', 'Cargo_Representante', 'Regiao_de_Comercializacao', 'Data_Registro_ANS', 'Data_Descredenciamento', 'Motivo_do_Descredenciamento']


In [13]:
#gera df operadoras ativas / canceladas em 2025

In [14]:
dfs_operadoras = []
for operador in operadoras:
    df = pd.read_csv(operador, sep=';', encoding='latin1')
    if 'Data_Descredenciamento' in df.columns:
        df['Data_Descredenciamento'] = pd.to_datetime(df['Data_Descredenciamento'], errors='coerce')
        df = df[df['Data_Descredenciamento'].isna() | (df['Data_Descredenciamento'].dt.year >= 2025)]
        dfs_operadoras.append(df)
    else:
        dfs_operadoras.append(df)

dfs_operadoras = pd.concat(dfs_operadoras, ignore_index=True)

dfs_operadoras.head()


,REGISTRO_OPERADORA,CNPJ,Razao_Social,Nome_Fantasia,Modalidade,Logradouro,Numero,Complemento,Bairro,Cidade,...,DDD,Telefone,Fax,Endereco_eletronico,Representante,Cargo_Representante,Regiao_de_Comercializacao,Data_Registro_ANS,Data_Descredenciamento,Motivo_do_Descredenciamento
0,419761,19541931000125,18 DE JULHO ADMINISTRADORA DE BENEFÃCIOS LTDA,NaN,Administradora de BenefÃ­cios,RUA CAPITÃO MEDEIROS DE REZENDE,274,NaN,PRAÃA DA BANDEIRA,AlÃ©m ParaÃ­ba,...,32.0,34624649.0,NaN,contabilidade@cbnassessoria.com.br,LUIZ HENRIQUE MARENDINO GONÃALVES,SÃCIO ADMINISTRADOR,6.0,2015-05-19,NaT,NaN
1,421545,22869997000153,2B ODONTOLOGIA OPERADORA DE PLANOS ODONTOLÃGI...,NaN,Odontologia de Grupo,RUA CATÃO,128,SALA 126,VILA ROMANA,SÃ£o Paulo,...,11.0,34415852.0,NaN,labmarisol@gmail.com,MARISOL BECHELLI,SÃCIO ADMINISTRADORA,4.0,2019-06-13,NaT,NaN
2,421421,27452545000195,2CARE OPERADORA DE SAÃDE LTDA.,NaN,Medicina de Grupo,RUA: BERNARDINO DE CAMPOS,230,1Âº ANDAR,CENTRO,Campinas,...,19.0,37901224.0,NaN,ans.plano@hospitalcare.com.br,RODRIGO PINHO RIBEIRO,REPRESENTANTE,5.0,2018-10-09,NaT,NaN
3,418030,13138885000131,A.P.S. ADMINISTRADORA DE BENEFÃCOS LTDA.,A.P.S. SAÃDE.,Administradora de BenefÃ­cios,RUA VOLUNTÃRIOS DA PÃTRIA,2525,CONJUNTO 143 - SALA 01,SANTANA,SÃ£o Paulo,...,11.0,45223468.0,NaN,diretoria@apssaude.com.br,PERCÃVEL GAETA,SÃ³CIO-ADMINISTRADOR E REPRESEN,4.0,2011-05-05,NaT,NaN
4,314668,17505793000101,ABERTTA SAÃDE - ASSOCIAÃÃO BENEFICENTE DOS ...,ABERTTA SAÃDE,AutogestÃ£o,AV. BERNARDO MONTEIRO,831,"Subsolo, 2Âº andar e 3Âº andar",SANTA EFIGÃNIA,Belo Horizonte,...,31.0,32484300.0,32484377.0,abertta.ans@arcelormittal.com.br,WERNER DUARTE DALLA,Diretor Presidente,4.0,1998-12-28,NaT,NaN


In [74]:
def gera_consolidado(df):
    df['Trimestre'] = df['DATA'].str.slice(5,7).map({'01':'1T','04':'2T','07':'3T'})
    df['Ano'] = df['DATA'].str.slice(0,4)
    df['VL_SALDO_FINAL'] = pd.to_numeric(df['VL_SALDO_FINAL'].str.replace(',', '.', regex=False), errors='coerce')
    df['VL_SALDO_INICIAL'] = pd.to_numeric(df['VL_SALDO_INICIAL'].str.replace(',', '.', regex=False), errors='coerce')
    df['ValorDespesas'] = df ['VL_SALDO_FINAL'] - df ['VL_SALDO_INICIAL']
    df_consolidado = df[['CNPJ', 'Razao_Social', 'Trimestre', 'Ano', 'ValorDespesas']]

    return df_consolidado

df_final = (
    df_eventos.merge(
    dfs_operadoras, 
    left_on='REG_ANS', 
    right_on='REGISTRO_OPERADORA', 
    how='inner'   
    )
    .pipe(gera_consolidado)
)

csv_path = os.path.join('data', 'consolidados_despesas.csv')
zip_path = os.path.join('data', 'consolidados_despesas.zip')

df_final.to_csv(csv_path, index=False, sep=';', encoding='latin1')

with open(csv_path, 'rb') as f_in:
    with open(zip_path, 'wb') as f_out:
        f_out.write(f_in.read())

df_final.head(10)

,CNPJ,Razao_Social,Trimestre,Ano,ValorDespesas
0,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,3558.06
1,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,0.00
2,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,-3300.99
3,42465310000121,TELOS - FUNDAÃÃO EMBRATEL DE SEGURIDADE SOCIAL,1T,2025,-3300.99
4,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,1T,2025,0.00
5,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,1T,2025,124677.97
6,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,1T,2025,124876.67
7,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,1T,2025,102223.76
8,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,1T,2025,102223.76
9,93507895000136,POLIMÃDICA SAÃDE SOCIEDADE SIMPLES LTDA,1T,2025,102223.76


In [ ]:
# FINAL TESTE 1 -> INICIO TESTE 2 
# Encontrados arquivos inconsistentes, com despesas negativas, zeradas, etc. 
# CNPJs duplicado também, trimestres aparentemente normais. 
# Solução: Corrigir. Já pegando o gancho do teste 2.1, vou validar valores 0/-,
# verificar trimestres e validar, corrigir cnpjs duplicados e com razoes sociais diferentes 
#  e aplicar uma validação de formato. Também verificar razão social não vazia.
#  TRADE-OFF DA VALIDAÇÃO: NO README.md

In [23]:
%pip install pandas validate-docbr

from validate_docbr import CNPJ

cnpj_tool = CNPJ()

def valida_valores_negativos_e_zeros(df):
    df_validado = df[(df['ValorDespesas'] > 0)]
    return df_validado

def valida_trimestres(df):
    trimestres_validos = {'1T', '2T', '3T'}
    df_validado = df[df['Trimestre'].isin(trimestres_validos)]
    return df_validado

def valida_cnpjs(df):
    
    #primeiro valida formato
    df['CNPJ'] = df['CNPJ'].astype(str)
    df = df[df['CNPJ'].apply(cnpj_tool.validate)]
    
    #valida duplicatas
    df = df.drop_duplicates(subset=['CNPJ', 'Trimestre'], keep='first')
    
    #valida razão social difernente para mesmo CNPJ
    df = df.sort_values(by=['CNPJ', 'Razao_Social'])
    df = df.drop_duplicates(subset=['CNPJ', 'Trimestre'], keep='first')
    
    #valida razao social vazia
    df = df[df['Razao_Social'].notna() & (df['Razao_Social'].str.strip() != '')]
    
    return df

df_final = (
    df_final.pipe(valida_valores_negativos_e_zeros)
            .pipe(valida_trimestres)
            .pipe(valida_cnpjs)
)

df_final.head(20)


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


,CNPJ,Razao_Social,Trimestre,Ano,ValorDespesas
53970,10219897000100,UNIMED OESTE DO PARÃ - COOPERATIVA DE TRABALH...,1T,2025,447765.09
103246,10219897000100,UNIMED OESTE DO PARÃ - COOPERATIVA DE TRABALH...,2T,2025,118132.06
117994,10219897000100,UNIMED OESTE DO PARÃ - COOPERATIVA DE TRABALH...,3T,2025,23628583.80
13576,10364053000145,ODONTOLIVE OPERADORA DE PLANOS ODONTOLÃGICOS ...,1T,2025,2581.24
78223,10364053000145,ODONTOLIVE OPERADORA DE PLANOS ODONTOLÃGICOS ...,2T,2025,22191.28
120625,10364053000145,ODONTOLIVE OPERADORA DE PLANOS ODONTOLÃGICOS ...,3T,2025,831.54
9284,10395358000114,UNIMED DO CEARÃ - FEDERAÃÃO DAS SOCIEDADES ...,1T,2025,178744.30
67895,10395358000114,UNIMED DO CEARÃ - FEDERAÃÃO DAS SOCIEDADES ...,2T,2025,100829.15
146057,10395358000114,UNIMED DO CEARÃ - FEDERAÃÃO DAS SOCIEDADES ...,3T,2025,267410.63
13020,10414182000109,UNIMED SAÃDE E ODONTO S.A,1T,2025,19055098.88


In [63]:
df_operadoras_ativas = (
    pd.read_csv(os.path.join('data', 'Relatorio_cadop.csv'), sep=';', encoding='latin1')   
)



def trata_cpnjs_duplicados(df1):
    df1['CNPJ'] = df1['CNPJ'].astype(str)
    
    df1['Data_Registro_ANS'] = pd.to_datetime(df1['Data_Registro_ANS'], errors='coerce')
    df_tratado = df1.sort_values(by=['Data_Registro_ANS']).drop_duplicates(subset=['CNPJ'], keep='last')   
    
    return df_tratado

df_operadoras_ativas_tratado = trata_cpnjs_duplicados(df_operadoras_ativas)

df_tratado = pd.merge(
    left=df_final, 
    right=df_operadoras_ativas_tratado, 
    on='CNPJ',
    how='inner' 
)

#lembrar trade-off 2.2

df_tratado = df_tratado.rename(columns={'Razao_Social_x': 'Razao_Social'})
df_tratado = df_tratado.rename(columns={'REGISTRO_OPERADORA': 'RegistroANS'})
df_tratado = df_tratado[['CNPJ', 'Razao_Social', 'Trimestre', 'Ano', 'ValorDespesas', 'RegistroANS', 'Modalidade', 'UF']]


df_tratado.head()

,CNPJ,Razao_Social,Trimestre,Ano,ValorDespesas,RegistroANS,Modalidade,UF
0,10219897000100,UNIMED OESTE DO PARÃ - COOPERATIVA DE TRABALH...,1T,2025,447765.09,362140,Cooperativa MÃ©dica,PA
1,10219897000100,UNIMED OESTE DO PARÃ - COOPERATIVA DE TRABALH...,2T,2025,118132.06,362140,Cooperativa MÃ©dica,PA
2,10219897000100,UNIMED OESTE DO PARÃ - COOPERATIVA DE TRABALH...,3T,2025,23628583.80,362140,Cooperativa MÃ©dica,PA
3,10364053000145,ODONTOLIVE OPERADORA DE PLANOS ODONTOLÃGICOS ...,1T,2025,2581.24,417831,Odontologia de Grupo,SP
4,10364053000145,ODONTOLIVE OPERADORA DE PLANOS ODONTOLÃGICOS ...,2T,2025,22191.28,417831,Odontologia de Grupo,SP


In [53]:
#Agrupar por Razao Social e UF

In [76]:
df_agregado = df_tratado.groupby(['Razao_Social', 'UF'], as_index=False)['ValorDespesas'].sum()

# tempo curto, mas para fazer a media é agrupar por trimestres, apos somar tudo de cada
# desvio padrao usa std()

print(df_agregado.shape)
df_agregado = df_agregado.sort_values(by='ValorDespesas', ascending=False)
df_agregado.head(20)

#colocar trade off 2.3: optei por sort_values pois o volume de dados não é significativo
# caso fosse, seguir abordagem dos chunks novamente. 

csv_path = os.path.join('data', 'despesas_agregadas.csv')
zip_path = os.path.join('data', 'Teste_Gabriel_Diniz.zip')

df_agregado.to_csv(csv_path, index=False, sep=';', encoding='latin1')

with open(csv_path, 'rb') as f_in:
    with open(zip_path, 'wb') as f_out:
        f_out.write(f_in.read())


(505, 3)
